# Data Processing Pipeline

## Purpose

This document tracks the exploration, processing, parsing and ultimate storage pattern of the [Bigfoot Sightings Dataset on kaggle](https://www.kaggle.com/datasets/josephvm/bigfoot-sightings-data)

## Prerequisites

* TODO:

In [5]:
import pandas as pd

df = pd.read_csv('raw/bigfoot-sightings-data.csv')
df.head()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5467 entries, 0 to 5466
Data columns (total 28 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   Report Type          5467 non-null   str  
 1   Id                   5467 non-null   int64
 2   Class                5016 non-null   str  
 3   Submitted Date       5467 non-null   str  
 4   Headline             5467 non-null   str  
 5   Year                 5015 non-null   str  
 6   Season               5016 non-null   str  
 7   Month                4398 non-null   str  
 8   State                5016 non-null   str  
 9   County               5016 non-null   str  
 10  Location Details     4254 non-null   str  
 11  Nearest Town         4694 non-null   str  
 12  Nearest Road         4310 non-null   str  
 13  Observed             4958 non-null   str  
 14  Also Noticed         3271 non-null   str  
 15  Other Witnesses      4463 non-null   str  
 16  Other Stories        3529 non-null 

### Noticed Patterns

We have 28 columns, with `str` data types dominating the data set.  We need to split these up into their various related entities, starting with the independent (non-foreign-key-bearing) entities before moving on to their dependents.  There is also the task of finding some way to classify thing's as duplicates and account for null values, as this will affect the cardinality of the entity sets we're putting these into.

We can't begin to answer these questions without collecting some basic statistics about the data.

In [6]:
# Basic null analysis
df.isnull().sum().sort_values(ascending=False)

Source Url             5326
A & G References       5227
Observed.1             5072
Media Issue            5062
Media Source           5023
Author                 5016
Also Noticed           2196
Date                   2028
Other Stories          1938
Follow-Up Report       1243
Follow-Up              1241
Location Details       1213
Nearest Road           1157
Month                  1069
Other Witnesses        1004
Time And Conditions     936
Nearest Town            773
Environment             725
Observed                509
Year                    452
Class                   451
State                   451
Season                  451
County                  451
Headline                  0
Report Type               0
Id                        0
Submitted Date            0
dtype: int64

In [7]:
# Cardinality (how many unique values in each column)
df.nunique().sort_values(ascending=False)

Headline               5440
Id                     5317
Observed               4928
Environment            4647
Time And Conditions    4422
Location Details       4176
Follow-Up Report       4159
Submitted Date         3909
Nearest Town           3776
Nearest Road           3749
Other Witnesses        3675
Also Noticed           2948
Other Stories          2913
Date                   1157
County                 1022
Year                    429
Media Issue             398
Observed.1              389
Media Source            354
Follow-Up               292
Author                  280
A & G References        221
Source Url              133
State                    49
Month                    12
Season                    5
Class                     3
Report Type               2
dtype: int64

### High-level Observations

* We have some columns (like `Source Url`) with a very high number of nulls
* Other columns have very few distinct values (low cardinality) such as `Class` and `Report Type`

Now we must start cross-referencing this against our schema and see where we land

In [ ]:
# Source 
df[['Media Source',\
    'Source Url',
    'Media Issue']]\
    .rename(columns={\
        'Media Source': 'Name',\
        'Source Url': 'Url',\
        'Media Issue': 'Issue'\
    }).drop_duplicates().shape
# Output: (440, 3)
# There are 440 unique media sources in the dataset

(440, 3)

In [25]:
# Location
df[[
    'State',
    'County',
    'Nearest Town',
    'Nearest Road'
    ]]\
    .drop_duplicates().shape

(4859, 4)

In [ ]:
df[[
    "Environment",
    "Time And Conditions"
]][df['Environment'].notnull() & df['Time And Conditions'].notnull()]\
    .rename(columns={\
        "Environment": "Environment_Description",\
        "Time And Conditions": "Time_And_Conditions"\
            })\
    .drop_duplicates()

,Environment_Description,Time_And_Conditions
0,"In the middle of the woods, in a clearing cove...",Middle of the night. The only light was the he...
1,"A pine forest, with a bog or swamp on the righ...","Started at 11, ended at about 3-3:30. Weather ..."
4,"Lake front,creek spit, gravel and sand, alder ...","Approximately 12:30 pm, partially coudy/sunny."
5,This sighting was located at approximately 1 t...,About 12:00 Midnight / full moon / clear / dim...
6,The forest was made up of mostly birch trees w...,around 6pm
8,"Upland Aspen/birch forest, above O'connor Creek.","Middle of the night, clear weather, thumbnail ..."
12,sighting of track was at the edge of a swamp. ...,Early morning overcast
14,Most of Prince of Wales is forest. There are a...,11:15 PM??
15,There were 2 bridges nearby and we were on a r...,It happend in the daytime and if I was to gues...
16,"Beach on the right side, with brush. A small m...","It was starting to get dark, we needed the lig..."


In [55]:
# Let's create the function that can map a table to an insert statement
def to_sql_insert(table, row, schema: dict):
    columns = []
    values = []
    
    for col in schema.keys():
        columns.append(col)
        values.append(format_value(row[col], schema[col]))
    
    columns_str = ', '.join(columns)
    values_str = ', '.join(values)
    
    return f"INSERT INTO {table} ({columns_str}) VALUES ({values_str});"

## We'll also handle the formatting of values based on their type
def format_value(value, data_type):
    if pd.isnull(value):
        return 'NULL'
    
    if data_type == 'int':
        try:
            return str(int(value))
        except:
            return 'NULL'
    
    elif data_type == 'str':
        safe = str(value).replace("'", "''")  # Escape single quotes
        return f"'{safe}'"

    elif data_type == 'date':
        try:
            date = pd.to_datetime(value).date()
            if pd.isna(date):
                return 'NULL'
            return f"'{date.strftime('%Y-%m-%d')}'"
        except:
            return 'NULL'
        
    else:
        return 'NULL'  # Default fallback for unsupported types

In [74]:
# We have a ton of data, so we may want to extract a small subset of it for the assignment
subset_df =df[df['Report Type'].notna() &
   df['Submitted Date'].notna() &
   df['Author'].notna() &
   df['Media Source'].notna() &
   df['Source Url'].notna() &
   df['Source Url'].str.match(r'^(https?://)?(www\.)?([a-zA-Z0-9-]+\.)+[a-zA-Z]{2,}(/.*)?$') &
   df['Media Issue'].notna()]
subset_df.count()

Report Type            112
Id                     112
Class                    0
Submitted Date         112
Headline               112
Year                     0
Season                   0
Month                    0
State                    0
County                   0
Location Details         0
Nearest Town             0
Nearest Road             0
Observed                 0
Also Noticed             0
Other Witnesses          0
Other Stories            0
Time And Conditions      0
Environment              0
Follow-Up                0
Follow-Up Report         0
Date                     0
Author                 112
Media Source           112
Source Url             112
Media Issue            112
Observed.1             108
A & G References         0
dtype: int64

In [75]:
source_df = subset_df[['Media Source',\
    'Source Url',
    'Media Issue']]\
    .rename(columns={\
        'Media Source': 'Name',\
        'Source Url': 'URL',\
        'Media Issue': 'Issue'\
            }).drop_duplicates()\
    .assign(Source_ID=lambda x: range(1, len(x) + 1))\
    [['Source_ID', 'Name', 'URL', 'Issue']]

source_schema = {
    'Source_ID': 'int',
    'Name': 'str',
    'URL': 'str',
    'Issue': 'str'
}
source_inserts = []

source_df

,Source_ID,Name,URL,Issue
3,1,The Delta Discovery,http://www.deltadiscovery.com/story/2013/01/23...,"Volume 15, Issue 4"
33,2,he Clanton Advertiser,http://www.clantonadvertiser.com/articles/2004...,Researches from around the country have contac...
34,3,BC13 Birmingham Tuscaloosa Alabama,http://www.nbc13.com/news/3508750/detail.html,For decades people there have been talking abo...
35,4,The Clanton Advertiser,http://www.clantonadvertiser.com/articles/2004...,It's been almost 25-years since the original r...
50,5,Dispatches from the LP-OP,http://leepeacock2010.blogspot.com/,"According to the witnesses, who spoke to The C..."
...,...,...,...,...
5299,106,travelwisconsin.com,http://www.travelwisconsin.com/article/natural...,This is the signature line to Finding Bigfoot...
5315,107,Baltimore Post-Examiner,http://baltimorepostexaminer.com/big-foot-disc...,Sheriff John Spears told the local press that ...
5316,108,The County Line,http://thecountyline.net/pages/?p=10858,"Of course, Bigfoot sightings have been around ..."
5325,109,Greater Milwaukee Today (www.GMToday.com),http://www.gmtoday.com/news/local_stories/2006...,Witness denies labeling large animal Bigfoot


In [76]:
source_inserts = [
    to_sql_insert('SOURCE', row, source_schema) for _, row in source_df.iterrows()
]

In [77]:
# Now lets make about 20 of these rows for the assignment

with open('source_inserts.sql', 'w') as f:
    for insert in source_inserts[:20]:
        f.write(insert + '\n')
    f.close()
    

In [78]:
report_schema: dict = {
    'Report_Type': 'str',
    'Headline': 'str',
    'Submitted_Date': 'date',
    'Author': 'str',
    'Source_ID': 'int'
}

report_df = subset_df[['Report Type', 'Headline', 'Submitted Date', 'Author']]\
    .assign(Source_ID=range(1, len(subset_df) + 1))\
    .rename(columns={\
        'Report Type': 'Report_Type',\
        'Submitted Date': 'Submitted_Date'})\
    .drop_duplicates()
    
report_df.head(2)

,Report_Type,Headline,Submitted_Date,Author,Source_ID
3,Media Article,Legendary Bigfoot sighted near Kasigluk,"Wednesday, January 23, 2013",By KJ Lincoln,1
33,Media Article,Bigfoot sightings of years past draw some inte...,"Sunday, July 11, 2004",T,2


In [67]:
report_inserts = [
    to_sql_insert('REPORT', row, report_schema) for _, row in report_df.iterrows()
]

report_inserts[:20]

["INSERT INTO REPORT (Report_Type, Headline, Submitted_Date, Author, Source_ID) VALUES ('Media Article', 'Legendary Bigfoot sighted near Kasigluk', '2013-01-23', 'By KJ Lincoln', 1);",
 "INSERT INTO REPORT (Report_Type, Headline, Submitted_Date, Author, Source_ID) VALUES ('Media Article', 'Bigfoot sightings of years past draw some interest now', '2004-07-11', 'T', 2);",
 "INSERT INTO REPORT (Report_Type, Headline, Submitted_Date, Author, Source_ID) VALUES ('Media Article', 'Bigfoot Legend Thrives In Chilton County: Man-Like Creature Lives In Peach Grove, According To Legend', '2004-07-08', 'N', 3);",
 "INSERT INTO REPORT (Report_Type, Headline, Submitted_Date, Author, Source_ID) VALUES ('Media Article', 'Bigfoot sighting reaching Silver milestone', '2004-07-08', 'By Jason Cannon', 4);",
 "INSERT INTO REPORT (Report_Type, Headline, Submitted_Date, Author, Source_ID) VALUES ('Media Article', 'Strange creature sighted near Sepulga River in Conecuh County, Alabama', '2016-07-28', 'By Lee P

In [80]:
with open('report_inserts.sql', 'w') as f:
    for insert in report_inserts[:20]:
        f.write(insert + '\n')
    f.close()

In [ ]:
import re
import random

follow_up = df['Follow-Up'].astype(str).dropna()

investigator_names = []
for text in follow_up:
    text = text.strip()
    match = re.search(r'BFRO Investigator\s+(.+?)\s*$', text, re.IGNORECASE)
    if match:
        investigator_names.append(match.group(1).strip())
        continue

    match = re.search(r'Follow-up report conducted by Investigator\s+(.+?)\.?$', text, re.IGNORECASE)
    if match:
        investigator_names.append(match.group(1).strip())
        continue

    match = re.search(r'Follow-up report by Investigator\s+(.+?)\.?$', text, re.IGNORECASE)
    if match:
        investigator_names.append(match.group(1).strip())
        continue

titles = ['Senior Investigator', 'Lead Investigator', 'Junior Investigator', 'Field Investigator', 'Research Investigator', 'Chief Investigator']
organizations = ['BFRO', 'Cryptozoology Society', 'Wildlife Research Institute', 'Bigfoot Field Research', 'Sighting Documentation Center', 'Cryptid Investigation Bureau']

investigator_df = (
    pd.DataFrame({'Investigator_Name': investigator_names})
    .drop_duplicates()
    .assign(Investigator_ID=lambda x: range(1, len(x) + 1))
    .assign(Title=lambda x: [random.choice(titles) for _ in range(len(x))])
    .assign(Organization=lambda x: [random.choice(organizations) for _ in range(len(x))])
    [['Investigator_ID', 'Investigator_Name', 'Title', 'Organization']]
)

investigator_schema = {
    'Investigator_ID': 'int',
    'Investigator_Name': 'str',
    'Title': 'str',
    'Organization': 'str'
}

investigator_inserts = [
    to_sql_insert('INVESTIGATOR', row, investigator_schema)
    for _, row in investigator_df.iterrows()
]

with open('investigator_inserts.sql', 'w') as f:
    for insert in investigator_inserts[:20]:
        f.write(insert + '\n')
